# Silver DPWH project component

## Purpose

Preserve every selected Bronze DPWH row as one Silver component row. Parse the fields that later steps need, and attach only approved category and status mappings. Repeated Contract IDs stay, because they may be components or funding-year records.

## Grain

One row per selected Bronze DPWH source row, per Silver run.

- Primary key: `component_key`.
- Source-row identity: `source_snapshot_id`, `source_row_hash`, and `source_row_ordinal`.
- `source_row_hash` hashes every Bronze business field that Silver reads. `source_row_ordinal` numbers rows that are identical on all of those fields. Identical rows are interchangeable, so the numbering is the same on every run.

## Inputs

- `01-bronze.dpwh_projects`, the selected snapshot only.
- `01-bronze.load_log` and `04-validation.dq_results`, for the safety gate.
- `02-silver.config_project_category_mapping` and `02-silver.config_project_status_mapping`, active `APPROVED` rows only.

## Outputs

- `02-silver.silver_dpwh_project_component`

## Not done here

- No row is dropped, merged, or deduplicated. `silver_project` consolidates by Contract ID.
- No category or status rule is written here. Unapproved values stay `UNMAPPED`.
- No project-to-region or project-to-flood matching. Tables 13 and 15 do that.

## 1. Confirm the Bronze safety gate

**What:** read the latest terminal DPWH load and its latest Bronze validation.

**Why:** Silver must never read a failed, partial, or unvalidated Bronze load. The latest terminal event wins, so a newer failure blocks an older success.

**Protects:** Timeliness and Auditability. The notebook stops before any write.

**Expected result:** one row with status `SUCCESS` or `SKIPPED_IDEMPOTENT`, a snapshot ID, and rows loaded above zero.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW latest_dpwh_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY
                COALESCE(completed_at, started_at) DESC,
                started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'dpwh_projects'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'STOP: latest DPWH Bronze load is not safe for Silver'
)
FROM latest_dpwh_load;

CREATE OR REPLACE TEMPORARY VIEW latest_dpwh_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.dq_results
    WHERE table_name = 'dpwh_projects'
      AND snapshot_id = (SELECT snapshot_id FROM latest_dpwh_load)
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest DPWH Bronze validation is unsafe'
)
FROM latest_dpwh_validation;

SELECT
    table_name,
    status,
    snapshot_id,
    rows_loaded
FROM latest_dpwh_load;

## 2. Declare the component parameters once

**What:** put every label, version, and threshold this notebook uses in one small view.

**Why:**

- `component_source_system = 'DPWH'` is the label every Silver DPWH table already publishes. Tables 13 and 15 join on it.
- `mapping_source_system = 'dpwh_projects'` is the key that DPWH rows use in the config tables. `04_validation/02_validation_silver_config` checks mapping coverage with this exact key, so reviewers approve rows under it.
- `mapping_source_infra_type = ''` is how the config contract stores "no infrastructure type". DPWH has no such column.
- The progress range comes from the required analytical validation in `docs/data-model.md`: 0 to 100 or `NULL`.
- `transformation_rule_version` changes whenever a rule in this notebook changes. The run ID includes it, so a new rule gives a new run.

**Protects:** no hard-coded values spread across cells. One edit changes one rule.

**Expected result:** one parameter row.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dpwh_component_parameters AS
SELECT
    'DPWH' AS component_source_system,
    'dpwh_projects' AS mapping_source_system,
    '' AS mapping_source_infra_type,
    'silver_dpwh_component_v2' AS transformation_rule_version,
    CAST(0 AS DOUBLE) AS progress_min_pct,
    CAST(100 AS DOUBLE) AS progress_max_pct;

SELECT * FROM dpwh_component_parameters;

## 3. Select approved mappings that cannot multiply rows

**What:** read only active `APPROVED` DPWH category and status rules, keyed the same way the config validator keys them.

**Why:**

- The earlier join matched categories on `raw_category` alone. It ignored `source_system` and `source_infra_type`, so one category with two approved rules would have copied a source row. The status join used `'DPWH'`, but approved status rows use `'dpwh_projects'`, so no status would ever map.
- The category table was empty and the status table had no rules when this was fixed, so the defect never reached output.
- Raw values are compared after `TRIM`, with blanks as `''`. This is the same normalization the config validator uses for coverage.

**Protects:** Uniqueness and Consistency. Each source row can match at most one rule, so a mapping can never duplicate a budget.

**Expected result:** both STOP checks pass. Empty mapping tables are valid. Every value then stays `UNMAPPED`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW approved_category_mappings AS
SELECT
    mapping.raw_category,
    mapping.standardized_sector,
    mapping.standardized_category,
    mapping.taxonomy_version
FROM `02-silver`.config_project_category_mapping AS mapping
INNER JOIN dpwh_component_parameters AS parameters
    ON mapping.source_system = parameters.mapping_source_system
    AND mapping.source_infra_type = parameters.mapping_source_infra_type
WHERE mapping.approval_status = 'APPROVED'
  AND mapping.is_active = TRUE;

CREATE OR REPLACE TEMPORARY VIEW approved_status_mappings AS
SELECT
    mapping.source_status,
    mapping.standardized_status,
    mapping.status_group,
    mapping.status_mapping_version
FROM `02-silver`.config_project_status_mapping AS mapping
INNER JOIN dpwh_component_parameters AS parameters
    ON mapping.source_system = parameters.mapping_source_system
WHERE mapping.approval_status = 'APPROVED'
  AND mapping.is_active = TRUE;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT taxonomy_version) <= 1
    AND COUNT(*) = COUNT(DISTINCT raw_category),
    'STOP: active approved DPWH category rules need one taxonomy version and one rule per raw category'
)
FROM approved_category_mappings;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT status_mapping_version) <= 1
    AND COUNT(*) = COUNT(DISTINCT source_status),
    'STOP: active approved DPWH status rules need one mapping version and one rule per source status'
)
FROM approved_status_mappings;

SELECT
    (SELECT COUNT(*) FROM approved_category_mappings) AS approved_category_rules,
    (SELECT COUNT(*) FROM approved_status_mappings) AS approved_status_rules;

## 4. Give every source row a stable identity and parse its fields

**What:** hash each row's business fields, number identical rows, and parse budget, progress, dates, coordinates, and infrastructure year. Each parsed field gets a quality status.

**Why:**

- Bronze has no source row number for DPWH. A hash of the fields Silver reads, plus an ordinal for identical rows, gives a key that is the same on every run. It needs no random ID and no file order.
- `TRY_CAST` keeps one bad value from stopping the run. The quality status then says why a parsed value is `NULL` or out of range: `MISSING`, `UNPARSEABLE`, `NEGATIVE`, or `OUT_OF_RANGE`.
- Out-of-range values stay as parsed. Silver flags them but does not change them, so reviewers can see shifted source values such as a year in the progress field.
- `infra_year` stays as the raw string. `infra_year_parsed` adds the integer, so Gold no longer has to cast it.

**Protects:** Uniqueness, Validity, and Auditability.

**Expected result:** the component row count equals the Bronze row count. The validator proves this.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dpwh_component_rows AS
WITH hashed_rows AS (
    SELECT
        bronze.*,
        SHA2(
            TO_JSON(
                NAMED_STRUCT(
                    'contract_id', bronze.contract_id,
                    'description', bronze.description,
                    'category', bronze.category,
                    'status', bronze.status,
                    'reported_budget', bronze.reported_budget,
                    'progress', bronze.progress,
                    'start_date', bronze.start_date,
                    'completion_date', bronze.completion_date,
                    'latitude', bronze.latitude,
                    'longitude', bronze.longitude,
                    'contractor', bronze.contractor,
                    'source_of_funds', bronze.source_of_funds,
                    'infra_year', bronze.infra_year,
                    'reported_region', bronze.reported_region
                )
            ),
            256
        ) AS source_row_hash
    FROM `01-bronze`.dpwh_projects AS bronze
),

identified_rows AS (
    SELECT
        hashed_rows.*,
        ROW_NUMBER() OVER (
            PARTITION BY hashed_rows.source_row_hash
            ORDER BY hashed_rows.source_row_hash
        ) AS source_row_ordinal
    FROM hashed_rows
),

parsed_rows AS (
    SELECT
        identified_rows.contract_id,
        parameters.component_source_system AS source_system,
        identified_rows.source_row_hash,
        identified_rows.source_row_ordinal,

        identified_rows.description AS project_description_raw,
        identified_rows.category AS category_raw,
        identified_rows.status AS status_raw,

        identified_rows.reported_budget AS reported_budget_raw,
        TRY_CAST(identified_rows.reported_budget AS DECIMAL(18, 2)) AS reported_budget_pesos,

        identified_rows.progress AS progress_raw,
        TRY_CAST(identified_rows.progress AS DOUBLE) AS physical_progress_pct,

        identified_rows.start_date AS start_date_raw,
        TRY_CAST(identified_rows.start_date AS DATE) AS start_date_parsed,

        identified_rows.completion_date AS completion_date_raw,
        TRY_CAST(identified_rows.completion_date AS DATE) AS completion_date_parsed,

        identified_rows.latitude AS latitude_raw,
        TRY_CAST(identified_rows.latitude AS DOUBLE) AS latitude_parsed,

        identified_rows.longitude AS longitude_raw,
        TRY_CAST(identified_rows.longitude AS DOUBLE) AS longitude_parsed,

        identified_rows.infra_year,
        TRY_CAST(TRIM(identified_rows.infra_year) AS INT) AS infra_year_parsed,

        identified_rows.contractor,
        identified_rows.source_of_funds,
        identified_rows.reported_region,

        identified_rows._source_snapshot_id,
        identified_rows._ingest_run_id,
        identified_rows._source_modified_at,

        parameters.transformation_rule_version,
        parameters.progress_min_pct,
        parameters.progress_max_pct
    FROM identified_rows
    CROSS JOIN dpwh_component_parameters AS parameters
)

SELECT
    SHA2(
        CONCAT_WS(
            '|',
            'DPWH_COMPONENT',
            parsed_rows._source_snapshot_id,
            parsed_rows.source_row_hash,
            CAST(parsed_rows.source_row_ordinal AS STRING)
        ),
        256
    ) AS component_key,

    parsed_rows.*,

    CASE
        WHEN NULLIF(TRIM(parsed_rows.reported_budget_raw), '') IS NULL THEN 'MISSING'
        WHEN parsed_rows.reported_budget_pesos IS NULL THEN 'UNPARSEABLE'
        WHEN parsed_rows.reported_budget_pesos < 0 THEN 'NEGATIVE'
        ELSE 'VALID'
    END AS budget_quality_status,

    CASE
        WHEN NULLIF(TRIM(parsed_rows.progress_raw), '') IS NULL THEN 'MISSING'
        WHEN parsed_rows.physical_progress_pct IS NULL THEN 'UNPARSEABLE'
        WHEN parsed_rows.physical_progress_pct < parsed_rows.progress_min_pct
            OR parsed_rows.physical_progress_pct > parsed_rows.progress_max_pct
            THEN 'OUT_OF_RANGE'
        ELSE 'VALID'
    END AS progress_quality_status,

    CASE
        WHEN NULLIF(TRIM(parsed_rows.infra_year), '') IS NULL THEN 'MISSING'
        WHEN parsed_rows.infra_year_parsed IS NULL THEN 'UNPARSEABLE'
        ELSE 'VALID'
    END AS infra_year_quality_status,

    CASE
        WHEN parsed_rows.latitude_raw IS NULL AND parsed_rows.longitude_raw IS NULL THEN 'MISSING_PAIR'
        WHEN parsed_rows.latitude_raw IS NULL OR parsed_rows.longitude_raw IS NULL THEN 'PARTIAL_PAIR'
        WHEN parsed_rows.latitude_parsed IS NULL OR parsed_rows.longitude_parsed IS NULL THEN 'UNPARSEABLE'
        ELSE 'VALID_PAIR'
    END AS coordinate_status,

    SHA2(
        CONCAT_WS('||', parsed_rows._source_snapshot_id, parsed_rows.transformation_rule_version),
        256
    ) AS run_id
FROM parsed_rows;

## 5. Publish one component row per Bronze row

**What:** attach the approved mappings and replace the table in one atomic write.

**Why:**

- `LEFT JOIN` keeps every source row, mapped or not. Step 3 proved each row can match at most one rule.
- `category_mapping_state` and `status_mapping_state` say `MAPPED` or `UNMAPPED`. Unmapped values keep their raw text and a `NULL` standardized value. The team has not yet decided whether Silver should publish the label `Unknown` instead. See the open question in `docs/silver_project_cleaning.md`.
- `CREATE OR REPLACE TABLE` makes a rerun replace the table rather than add rows.

**Protects:** Completeness and Consistency. No row is dropped, and mapped rows always carry their mapping version.

**Expected result:** a Delta table with one row per Bronze row.

In [ ]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_dpwh_project_component
USING DELTA
COMMENT 'One selected Bronze DPWH source row per Silver run. Raw and parsed values side by side, quality statuses, approved mappings only, and full lineage.'
AS
SELECT
    component.component_key,
    component.contract_id,
    component.source_system,
    component.source_row_hash,
    component.source_row_ordinal,

    component.project_description_raw,

    component.category_raw,
    category.standardized_sector,
    category.standardized_category,
    category.taxonomy_version,
    CASE WHEN category.raw_category IS NULL THEN 'UNMAPPED' ELSE 'MAPPED' END AS category_mapping_state,

    component.status_raw,
    status.standardized_status,
    status.status_group,
    status.status_mapping_version,
    CASE WHEN status.source_status IS NULL THEN 'UNMAPPED' ELSE 'MAPPED' END AS status_mapping_state,

    component.reported_budget_raw,
    component.reported_budget_pesos,
    component.budget_quality_status,

    component.progress_raw,
    component.physical_progress_pct,
    component.progress_quality_status,

    component.start_date_raw,
    component.start_date_parsed,
    component.completion_date_raw,
    component.completion_date_parsed,

    component.latitude_raw,
    component.latitude_parsed,
    component.longitude_raw,
    component.longitude_parsed,
    component.coordinate_status,

    component.infra_year,
    component.infra_year_parsed,
    component.infra_year_quality_status,

    component.contractor,
    component.source_of_funds,
    component.reported_region,

    component._source_snapshot_id,
    component._ingest_run_id,
    component._source_modified_at,

    component.run_id,
    component.transformation_rule_version
FROM dpwh_component_rows AS component
LEFT JOIN approved_category_mappings AS category
    ON COALESCE(TRIM(component.category_raw), '') = category.raw_category
LEFT JOIN approved_status_mappings AS status
    ON COALESCE(TRIM(component.status_raw), '') = status.source_status;

# Summary

- Created `02-silver.silver_dpwh_project_component` at one row per selected Bronze DPWH row, keyed by `component_key`.
- Every row has a stable source-row identity, a deterministic `run_id`, and its rule version.
- Category and status rules now join on the same key the config validator checks, so an approved rule can never copy a row.
- Budget, progress, and infrastructure-year values keep a quality status. Out-of-range and unparseable values stay visible as findings.
- Known limits: no category rule is approved yet, so every category stays `UNMAPPED`. Status uses approved mapping `2026-10-v1`. Blank or malformed statuses stay `UNMAPPED`. Long-running and stalled rules still wait on team decisions.
- Next: `02_silver/05_silver_project`, then `04_validation/04_validation_silver_projects`.

The query below shows coverage. Each count has its denominator, `component_rows`.

In [ ]:
%sql
SELECT
    COUNT(*) AS component_rows,
    COUNT(DISTINCT contract_id) AS distinct_contract_ids,
    COUNT_IF(source_row_ordinal > 1) AS repeated_identical_rows,
    COUNT_IF(category_mapping_state = 'MAPPED') AS category_mapped_rows,
    COUNT_IF(status_mapping_state = 'MAPPED') AS status_mapped_rows,
    COUNT_IF(budget_quality_status <> 'VALID') AS budget_finding_rows,
    COUNT_IF(progress_quality_status <> 'VALID') AS progress_finding_rows,
    COUNT_IF(infra_year_quality_status <> 'VALID') AS infra_year_finding_rows,
    COUNT_IF(coordinate_status <> 'VALID_PAIR') AS coordinate_finding_rows
FROM `02-silver`.silver_dpwh_project_component;